### Tugas 4 DSS - Coding 2 cases Perceptron
Nama: Felicia Stevany Lewa

NIM: 0706022210046

1. Berdasarkan penjelasan Algoritma Perceptron di kelas, termasuk semua property terkait -- neuron dan bobot bias; linear dan non-linear separable problem; binary dan bipolar representation -- lakukan coding Python untuk salah satu fungsi logika selain AND.


In [16]:
import numpy as np

def bipolar_activation(yin, theta):
    return 1 if yin >= theta else -1

def print_epoch_header():
    print(" X1  X2  Target    Yin   Yout  Error   Δw1   Δw2    Δb    w1    w2    b")

def perceptron_with_table(inputs, targets, alpha, theta, max_epochs=100):
    n_samples, n_features = inputs.shape
    weights = np.zeros(n_features)
    bias = 0.0

    for epoch in range(max_epochs):
        print(f"\nEpoch {epoch + 1}")
        print_epoch_header()
        total_error = 0
        for i in range(n_samples):
            x = inputs[i]
            target = targets[i]
            yin = np.dot(weights, x) + bias
            yout = bipolar_activation(yin, theta)
            error = target - yout
            delta_w = alpha * error * x
            delta_b = alpha * error

            # Update weights & bias
            weights += delta_w
            bias += delta_b

            total_error += abs(error)

            print(f"{x[0]:>3} {x[1]:>3}   {target:>5} {yin:>6.1f} {yout:>6} {error:>6} "
                  f"{delta_w[0]:>5.1f} {delta_w[1]:>5.1f} {delta_b:>5.1f} "
                  f"{weights[0]:>5.1f} {weights[1]:>5.1f} {bias:>5.1f}")

        if total_error == 0:
            print(f"\nTraining selesai di epoch {epoch + 1}")
            break

    # Cetak hasil akhir (opsional)
    data = list(zip(inputs.tolist(), targets.tolist()))
    print(data)
    return weights, bias

X = np.array([[-1, -1], [-1, 1], [1, -1], [1, 1]])
y = np.array([-1, 1, 1, 1])

alpha = 1
theta = 0

perceptron_with_table(X, y, alpha, theta)


Epoch 1
 X1  X2  Target    Yin   Yout  Error   Δw1   Δw2    Δb    w1    w2    b
 -1  -1      -1    0.0      1     -2   2.0   2.0  -2.0   2.0   2.0  -2.0
 -1   1       1   -2.0     -1      2  -2.0   2.0   2.0   0.0   4.0   0.0
  1  -1       1   -4.0     -1      2   2.0  -2.0   2.0   2.0   2.0   2.0
  1   1       1    6.0      1      0   0.0   0.0   0.0   2.0   2.0   2.0

Epoch 2
 X1  X2  Target    Yin   Yout  Error   Δw1   Δw2    Δb    w1    w2    b
 -1  -1      -1   -2.0     -1      0   0.0   0.0   0.0   2.0   2.0   2.0
 -1   1       1    2.0      1      0   0.0   0.0   0.0   2.0   2.0   2.0
  1  -1       1    2.0      1      0   0.0   0.0   0.0   2.0   2.0   2.0
  1   1       1    6.0      1      0   0.0   0.0   0.0   2.0   2.0   2.0

Training selesai di epoch 2
[([-1, -1], -1), ([-1, 1], 1), ([1, -1], 1), ([1, 1], 1)]


(array([2., 2.]), np.float64(2.0))

2. Ulangi point (1) di atas untuk weather (playing tennis) dataset. Gunakan DUA KALI percobaan, yang masing-masing memakai representasi binary dan bipolar untuk inputnya. Nilai alpha dan theta dapat dientry melalui keyboard. Hati-hatilah pengkodean input-attributes yang memiliki 3 values. Jelaskan semua pengkodean representasi input dan output yang Anda lakukan.

In [80]:
import pandas as pd
import numpy as np

def bipolar_activation(yin, theta):
    return 1 if yin >= theta else -1

def binary_activation(yin, theta):
    return 1 if yin >= theta else 0

def encode_data(df, bipolar=False):
    encoded = []

    outlook_map = {
        'sunny': [1, 0, 0] if not bipolar else [1, -1, -1],
        'overcast': [0, 1, 0] if not bipolar else [-1, 1, -1],
        'rainy': [0, 0, 1] if not bipolar else [-1, -1, 1],
    }

    temperature_map = {
        'hot': [1, 0] if not bipolar else [1, -1],
        'mild': [0, 1] if not bipolar else [-1, 1],
        'cool': [0, 0] if not bipolar else [-1, -1],
    }

    humidity_map = {
        'high': 1 if not bipolar else 1,
        'normal': 0 if not bipolar else -1,
    }

    windy_map = {
        'true': 1 if not bipolar else 1,
        'false': 0 if not bipolar else -1,
    }

    output_map = {
        'yes': 1,
        'no': 0 if not bipolar else -1,
    }

    for _, row in df.iterrows():
        row_vector = []
        row_vector.extend(outlook_map[row['outlook']])
        row_vector.extend(temperature_map[row['temp']])
        row_vector.append(humidity_map[row['humidity']])
        row_vector.append(windy_map[row['windy']])
        encoded.append(row_vector)

    X = np.array(encoded)
    y = np.array([output_map[label] for label in df['play']])
    return X, y

def perceptron_epoch_table(inputs, targets, alpha, theta, bipolar=False, max_epochs=100):
    n_samples, n_features = inputs.shape
    weights = np.zeros(n_features)
    bias = 0.0

    for epoch in range(max_epochs):
        print(f"\nEpoch {epoch + 1}")
        header = ("    " + "    ".join([f"X{i+1}" for i in range(n_features)]) + " Target    Yin  Yout  Error   " +
              "   ".join([f"Δw{i+1}" for i in range(n_features)]) + "   Δb   " +
              "    ".join([f"w{i+1}" for i in range(n_features)]) + "     b   ")
        print(header)
        print("-" * len(header))
        total_error = 0
        for i in range(n_samples):
            x = inputs[i]
            target = targets[i]
            yin = np.dot(weights, x) + bias
            yout = bipolar_activation(yin, theta) if bipolar else binary_activation(yin, theta)
            error = target - yout
            delta_w = alpha * error * x
            delta_b = alpha * error

            weights += delta_w
            bias += delta_b
            total_error += abs(error)

            # Print each row
            print(" " + " ".join(f"{val:>5}" for val in x) + f"  {target:>5} {yin:>6.1f} {yout:>5} {error:>6} " +
                  " ".join(f"{dw:>5.1f}" for dw in delta_w) + f" {delta_b:>4.1f} " +
                  " ".join(f"{w:>5.1f}" for w in weights) + f" {bias:>5.1f}")

        if total_error == 0:
            print(f"\nTraining selesai di epoch {epoch + 1}")
            break

    return weights, bias

# Baca dataset
df = pd.read_csv("https://raw.githubusercontent.com/feliciastevany/DSS-dataset/refs/heads/main/playtennis.csv")

# Mengonversi kolom 'windy' menjadi string
df['windy'] = df['windy'].astype(str).str.lower()

# Input alpha dan theta dari user
alpha = 1
theta = 0.8

print("\n--- Percobaan 1: Binary Representation ---")
X_bin, y_bin = encode_data(df, bipolar=False)
perceptron_epoch_table(X_bin, y_bin, alpha, theta, bipolar=False)

print("\n\n--- Percobaan 2: Bipolar Representation ---")
X_bip, y_bip = encode_data(df, bipolar=True)
perceptron_epoch_table(X_bip, y_bip, alpha, theta, bipolar=True)



--- Percobaan 1: Binary Representation ---

Epoch 1
    X1    X2    X3    X4    X5    X6    X7 Target    Yin  Yout  Error   Δw1   Δw2   Δw3   Δw4   Δw5   Δw6   Δw7   Δb   w1    w2    w3    w4    w5    w6    w7     b   
----------------------------------------------------------------------------------------------------------------------------------------------------------------------
     1     0     0     1     0     1     0      0    0.0     0      0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
     1     0     0     1     0     1     1      0    0.0     0      0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
     0     1     0     1     0     1     0      1    0.0     0      1   0.0   1.0   0.0   1.0   0.0   1.0   0.0  1.0   0.0   1.0   0.0   1.0   0.0   1.0   0.0   1.0
     0     0     1     0     1     1     0      1    2.0     1      0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  0.0  

(array([ -8.,   8.,  -4.,   0.,   8., -12.,  -8.]), np.float64(4.0))